# 02 · ¿Cambia según quién sea el competidor?

**Objetivo:** Mostrar a directores y dueños si la competencia cercana cambia el precio de una tienda, como caso de competencia local para decisiones de expansión y de precio por tienda.

**Pregunta:** ¿Importa si el competidor cercano es de Walmart de México, Soriana, Chedraui u otra cadena?

Importa porque un retailer no compite igual contra una bodega de descuento que contra un supermercado regional.
Indicadores de cada tipo de competidor a 2 km (una tienda puede tener varios); mismos controles.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import competencia as K
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
REPS = 500
foc = pd.read_csv(PROC / "tiendas_focales.csv")
ev = foc[foc.grupo.isin(K.EVALUABLES)].copy()
nt = pd.DataFrame({t: (ev[f"d_{t}"] <= 2).sum() for t in ["walmex", "soriana", "chedraui", "otro"]}, index=["tiendas con ese competidor a 2 km"])
display(nt)
ty = K.estimate(ev, K.comp_by_type(), reps=REPS)
display(ty.round(2))

,walmex,soriana,chedraui,otro
tiendas con ese competidor a 2 km,58,64,43,68


,diferencia_pct,ic90_inf,ic90_sup,solido
competidor walmex,0.63,-0.09,1.30,False
competidor soriana,0.03,-0.43,0.42,False
competidor chedraui,-0.27,-0.83,0.13,False
competidor otro,0.49,0.04,1.07,True


In [2]:
ty.to_csv(PROC / "efecto_por_tipo.csv", encoding="utf-8-sig")
F = "02_tipo_de_competidor"
for k_, r_ in ty.iterrows():
    t_ = k_.split()[-1]
    reg(f"tipo_{t_}", r_.diferencia_pct, [r_.ic90_inf, r_.ic90_sup], n=int((ev[f"d_{t_}"] <= 2).sum()), fuente=F)
reg("tipo_solidos", int(ty.solido.sum()), n=len(ty), fuente=F)
display(Markdown(
    "**Conclusión.** Ningún tipo de competidor se asocia con precios más bajos de forma clara: " +
    "; ".join(f"{k_.replace('competidor ', '')} {r_.diferencia_pct:+.1f}% (IC90 {r_.ic90_inf:+.1f} a {r_.ic90_sup:+.1f})" for k_, r_ in ty.iterrows()) + "."))

**Conclusión.** Ningún tipo de competidor se asocia con precios más bajos de forma clara: walmex +0.6% (IC90 -0.1 a +1.3); soriana +0.0% (IC90 -0.4 a +0.4); chedraui -0.3% (IC90 -0.8 a +0.1); otro +0.5% (IC90 +0.0 a +1.1).